# Home Credit - Preprocessing & Feature Engineering

This notebook builds model-ready datasets from all Home Credit tables.

Outputs:
- data/processed/train_features.parquet
- data/processed/test_features.parquet
- data/processed/parquet_cache/*.parquet (raw table cache for faster reloads)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PARQUET_CACHE = PROCESSED / 'parquet_cache'
PROCESSED.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE.mkdir(parents=True, exist_ok=True)

print('Preprocessing setup complete (Parquet cache enabled)')

Preprocessing setup complete (Parquet cache enabled)


## 1. Load Datasets

In [2]:
def load_table_with_cache(file_name):
    csv_path = RAW / f'{file_name}.csv'
    pq_path = PARQUET_CACHE / f'{file_name}.parquet'

    if pq_path.exists():
        return pd.read_parquet(pq_path)

    df = pd.read_csv(csv_path)
    df.to_parquet(pq_path, index=False)
    return df


application_train = load_table_with_cache('application_train')
application_test = load_table_with_cache('application_test')
bureau = load_table_with_cache('bureau')
bureau_balance = load_table_with_cache('bureau_balance')
credit_card = load_table_with_cache('credit_card_balance')
installments = load_table_with_cache('installments_payments')
pos_cash = load_table_with_cache('POS_CASH_balance')
previous_app = load_table_with_cache('previous_application')

print('Loaded all raw tables (using Parquet cache where available)')
print('application_train:', application_train.shape)
print('application_test :', application_test.shape)

Loaded all raw tables (using Parquet cache where available)
application_train: (307511, 122)
application_test : (48744, 121)


## 2. Helper Aggregation Functions

In [3]:
def replace_days_sentinels(df):
    df = df.copy()
    day_cols = [c for c in df.columns if c.startswith('DAYS_')]
    if day_cols:
        df[day_cols] = df[day_cols].replace(365243, np.nan)
    return df


def one_hot_encode(df, exclude_cols=None):
    exclude_cols = set(exclude_cols or [])
    cat_cols = [
        c for c in df.columns
        if c not in exclude_cols and (df[c].dtype == 'object' or str(df[c].dtype) == 'category')
    ]
    if not cat_cols:
        return df
    return pd.get_dummies(df, columns=cat_cols, dummy_na=True, dtype=np.int8)


def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator


def add_application_features(df):
    df = replace_days_sentinels(df)
    ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    existing_ext_cols = [c for c in ext_cols if c in df.columns]

    feature_map = {
        'CREDIT_TO_INCOME_RATIO': safe_divide(df['AMT_CREDIT'], df['AMT_INCOME_TOTAL']),
        'ANNUITY_TO_INCOME_RATIO': safe_divide(df['AMT_ANNUITY'], df['AMT_INCOME_TOTAL']),
        'CREDIT_TERM': safe_divide(df['AMT_CREDIT'], df['AMT_ANNUITY']),
        'GOODS_TO_CREDIT_RATIO': safe_divide(df['AMT_GOODS_PRICE'], df['AMT_CREDIT']),
        'EMPLOYED_TO_AGE_RATIO': safe_divide(df['DAYS_EMPLOYED'].abs(), df['DAYS_BIRTH'].abs()),
    }

    if existing_ext_cols:
        feature_map['EXT_SOURCE_MEAN'] = df[existing_ext_cols].mean(axis=1)
        feature_map['EXT_SOURCE_STD'] = df[existing_ext_cols].std(axis=1)
        feature_map['EXT_SOURCE_MISSING_COUNT'] = df[existing_ext_cols].isna().sum(axis=1)

    return df.assign(**feature_map)


def prepare_for_aggregation(df, exclude_cols=None):
    df = replace_days_sentinels(df)
    return one_hot_encode(df, exclude_cols=exclude_cols)


def safe_group_aggregate(df, key_col='SK_ID_CURR', prefix='SRC'):
    df = df.copy()
    if key_col not in df.columns:
        raise ValueError(f'{key_col} not found in dataframe')

    num_cols = [
        c for c in df.select_dtypes(include=['number']).columns
        if c not in [key_col, 'SK_ID_BUREAU', 'SK_ID_PREV', 'TARGET']
    ]

    agg_map = {c: ['mean', 'max', 'min', 'sum'] for c in num_cols}
    out = df.groupby(key_col).agg(agg_map)
    out.columns = [f'{prefix}_{c}_{stat}'.upper() for c, stat in out.columns]
    out.reset_index(inplace=True)

    out[f'{prefix}_RECORD_COUNT'] = df.groupby(key_col).size().values
    return out

## 3. Prepare Secondary Tables

In [4]:
application_train = add_application_features(application_train)
application_test = add_application_features(application_test)

# Merge bureau_balance to bureau at SK_ID_BUREAU level, then aggregate to SK_ID_CURR
bb = one_hot_encode(replace_days_sentinels(bureau_balance), exclude_cols=['SK_ID_BUREAU'])
bb_agg = bb.groupby('SK_ID_BUREAU').mean().reset_index()

bureau_enriched = replace_days_sentinels(bureau).merge(bb_agg, on='SK_ID_BUREAU', how='left')
bureau_enriched = prepare_for_aggregation(bureau_enriched, exclude_cols=['SK_ID_CURR', 'SK_ID_BUREAU'])
credit_encoded = prepare_for_aggregation(credit_card, exclude_cols=['SK_ID_CURR', 'SK_ID_PREV'])
pos_encoded = prepare_for_aggregation(pos_cash, exclude_cols=['SK_ID_CURR', 'SK_ID_PREV'])
prev_encoded = prepare_for_aggregation(previous_app, exclude_cols=['SK_ID_CURR', 'SK_ID_PREV'])
installments_clean = replace_days_sentinels(installments)

bureau_features = safe_group_aggregate(bureau_enriched, key_col='SK_ID_CURR', prefix='BUREAU')
credit_features = safe_group_aggregate(credit_encoded, key_col='SK_ID_CURR', prefix='CC')
inst_features = safe_group_aggregate(installments_clean, key_col='SK_ID_CURR', prefix='INST')
pos_features = safe_group_aggregate(pos_encoded, key_col='SK_ID_CURR', prefix='POS')
prev_features = safe_group_aggregate(prev_encoded, key_col='SK_ID_CURR', prefix='PREV')

print('Aggregations completed')
print('bureau_features:', bureau_features.shape)
print('credit_features:', credit_features.shape)
print('inst_features  :', inst_features.shape)
print('pos_features   :', pos_features.shape)
print('prev_features  :', prev_features.shape)

Aggregations completed
bureau_features: (305811, 90)
credit_features: (103558, 82)
inst_features  : (339587, 26)
pos_features   : (337252, 22)
prev_features  : (338857, 78)


## 4. Build Final Train/Test Feature Tables

In [5]:
train_df = application_train.copy()
test_df = application_test.copy()

for feat in [bureau_features, credit_features, inst_features, pos_features, prev_features]:
    train_df = train_df.merge(feat, on='SK_ID_CURR', how='left')
    test_df = test_df.merge(feat, on='SK_ID_CURR', how='left')

print('train_df:', train_df.shape)
print('test_df :', test_df.shape)

train_df: (307511, 415)
test_df : (48744, 414)


## 5. Missing Value Handling and Type Cleanup

In [6]:
# Replace inf with nan
train_df = train_df.replace([np.inf, -np.inf], np.nan)
test_df = test_df.replace([np.inf, -np.inf], np.nan)

# Drop columns with >80% missing in train
missing_ratio = train_df.isna().mean()
drop_cols = missing_ratio[missing_ratio > 0.80].index.tolist()
drop_cols = [c for c in drop_cols if c not in ['TARGET', 'SK_ID_CURR']]

train_df = train_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])

print(f'Dropped {len(drop_cols)} columns with >80% missing values')
print('train_df:', train_df.shape)
print('test_df :', test_df.shape)

Dropped 27 columns with >80% missing values
train_df: (307511, 388)
test_df : (48744, 387)


## 6. Save Processed Datasets

In [7]:
train_out = PROCESSED / 'train_features.parquet'
test_out = PROCESSED / 'test_features.parquet'

train_df.to_parquet(train_out, index=False)
test_df.to_parquet(test_out, index=False)

print('Saved:')
print(train_out)
print(test_out)

Saved:
..\data\processed\train_features.parquet
..\data\processed\test_features.parquet


## 7. Quick Quality Checks

In [8]:
print('TARGET distribution after preprocessing:')
print(train_df['TARGET'].value_counts(normalize=True).round(4))

print('\nTop 10 columns with missing% in final train set:')
print((train_df.isna().mean().sort_values(ascending=False).head(10) * 100).round(2))

print('\nSentinel check (DAYS_EMPLOYED == 365243) after preprocessing:')
print(int((train_df['DAYS_EMPLOYED'] == 365243).sum()))

print('\nSample engineered columns present:')
engineered_cols = [
    'EXT_SOURCE_MEAN',
    'EXT_SOURCE_STD',
    'CREDIT_TO_INCOME_RATIO',
    'ANNUITY_TO_INCOME_RATIO',
    'CREDIT_TERM',
    'EMPLOYED_TO_AGE_RATIO',
]
print([c for c in engineered_cols if c in train_df.columns])

TARGET distribution after preprocessing:
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64

Top 10 columns with missing% in final train set:
BUREAU_AMT_ANNUITY_MAX            73.98
BUREAU_AMT_ANNUITY_MEAN           73.98
BUREAU_AMT_ANNUITY_MIN            73.98
CC_AMT_INST_MIN_REGULARITY_MAX    71.74
CC_MONTHS_BALANCE_SUM             71.74
CC_MONTHS_BALANCE_MIN             71.74
CC_MONTHS_BALANCE_MAX             71.74
CC_MONTHS_BALANCE_MEAN            71.74
CC_AMT_INST_MIN_REGULARITY_SUM    71.74
CC_AMT_BALANCE_MAX                71.74
dtype: float64
